# Text → Image Baseline: Data, Embeddings, and 5 Test Runs
**Pipeline:** Flickr30k subset → clean → CLIP text embeddings (+ retrieval sanity check) → Stable Diffusion v1.5 prompt→image runs.

Run top-to-bottom. A GPU (Colab T4 is enough) makes cell 6 take ~1–2 min; CPU works but is slow.

## 1. Setup

In [ ]:
import os, sys, subprocess
# --- Colab only: clone the repo and install deps. Skipped when running locally from the repo root.
if "google.colab" in sys.modules and not os.path.exists("src"):
    REPO_URL = "https://github.com/<your-username>/genai-text2img.git"   # <- edit
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from src import data_prep, embed, generate
print("device:", embed.get_device())

## 2. Choose dataset
**Flickr30k** (31k images × 5 human captions; everyday scenes with people and animals). Streamed from the Hugging Face Hub, so only the images we keep are downloaded. Flickr30k is licensed for non-commercial research use.

## 3. Clean & preprocess 2,000 image–caption pairs

In [ ]:
N_IMAGES = 2000          # any value in 1,000–5,000 satisfies the assignment
df = data_prep.main(["--source", "hf", "--n-images", str(N_IMAGES), "--out", "data/processed"])
print(json.dumps(json.load(open("data/processed/stats.json")), indent=2))

In [ ]:
# Visual spot-check of the cleaned data
sample = df.sample(8, random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open("data/processed/" + r.image_path)); ax.axis("off")
    ax.set_title("\n".join(__import__("textwrap").wrap(r.caption, 36)), fontsize=8)
plt.tight_layout(); plt.show()
df.n_words.hist(bins=range(4, 42)); plt.xlabel("caption length (words)"); plt.title("Caption length distribution"); plt.show()

## 4. Environment check (PyTorch + Diffusers + CLIP)

In [ ]:
import torch, diffusers, transformers
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

## 5. Baseline text → embedding pipeline
CLIP ViT-B/32 embeds every caption and image into a shared 512-d space. As a sanity check we measure how often a caption retrieves its *own* image (Recall@k). We also extract the 77×768 per-token conditioning that Stable Diffusion's U-Net consumes.

In [ ]:
metrics = embed.run("data/processed/pairs.csv", "data/embeddings")
pd.DataFrame({k: v for k, v in metrics.items() if isinstance(v, dict)}).T

In [ ]:
sd_emb = embed.sd_prompt_embeddings(generate.TEST_PROMPTS)
print("Stable Diffusion conditioning tensor:", sd_emb.shape)   # (5, 77, 768)

## 6. Five prompt → image test runs
Fixed seeds (42–46), DPM-Solver++ sampler, guidance 7.5. Images, a contact sheet, and `results.json` (with CLIP scores) are saved to `outputs/test_runs/` and committed to the repo.

In [ ]:
images, meta = generate.run(out_dir="outputs/test_runs")

In [ ]:
Image.open("outputs/test_runs/grid.png")

In [ ]:
pd.DataFrame(meta["runs"])[["run", "prompt", "seed", "steps", "clip_score", "seconds", "safety_checker_flagged"]]

## 7. Next steps (Milestone 2)
Fine-tune the text-conditioning path (e.g., LoRA on the U-Net with the cleaned captions), compare against this baseline using CLIP score and retrieval Recall@k, and run an ablation on caption length and guidance scale.